# Module 38.4 — Evaluation



---

## 1. What is Model Evaluation?

Model evaluation is the process of measuring how well a trained image classification model performs on unseen data.

You have already studied accuracy, precision, recall, F1-score, and confusion matrices in Deep Learning. We'll reuse that knowledge and focus on how to interpret these metrics in real computer vision applications.

Consider a manufacturing inspection model that classifies motor components into:

- Normal
- Crack
- Surface defect
- Deformation

A model achieving 95% accuracy might sound excellent. But what if it misses most of the defective components?

That is why computer vision evaluation needs more than a single metric.

![AI for defect detection on a manufacturing production line](https://images.openai.com/static-rsc-4/4fTLjDe_b0eVPvqxhGm4EbyMq11eTs5Is2cY7JOvkiT-L9cYFAtZU3WXmqiZm9bBZqwyTCQn5MdlhYghEAmED36MzofINvohf6lIzMbjtJMj-lEEfMxXqq4noOgja1NkH1_RkZrvLO6IBM7qk-BOHUfFKrhhoc99v8C2NMgkk_s?purpose=inline)

[Source: Manufacturing Mag — AI for Defect Detection](https://www.manufacturingmag.com/article/ai-defect-detection)

---

## 2. Existing Deep Learning Metrics — Quick Revision

### A. Accuracy

Accuracy measures the proportion of all predictions that are correct.

$$
Accuracy = \frac{TP+TN}{TP+TN+FP+FN}
$$

Useful when classes are reasonably balanced and different error types have similar importance.

### B. Precision

Precision measures how many predicted positive samples are actually positive.

$$
Precision = \frac{TP}{TP+FP}
$$

Manufacturing example: Of all components flagged as defective, how many were actually defective?

High precision means fewer false alarms.

### C. Recall

Recall measures how many actual positive samples the model successfully identifies.

$$
Recall = \frac{TP}{TP+FN}
$$

Manufacturing example: Of all genuinely defective components, how many did the model detect?

High recall means fewer missed defects.

### D. F1-score

F1-score is the harmonic mean of precision and recall.

$$
F1 = 2\times\frac{Precision\times Recall}{Precision+Recall}
$$

Useful when you need a balance between false alarms and missed defects.

### E. Confusion Matrix

A confusion matrix compares actual labels with predicted labels.

![Confusion matrix illustration](https://images.openai.com/static-rsc-4/KMCjl-7rIebkBIcmBZlf1Ydf5Dy5dunBHCgD9hOII2UZJFjsTtwF7tR-IeQkaMi8KT-ZDYyJ3_U5uKW4OAEQEtxWaT9WrD9A9s268rvDuUQIUOjsTdDGJgSNI83Ue-DpFPKdxuHmAz3oxUksIpv0B5e8Kkye3TF1ujlTU7ns15A?purpose=inline)

[Source: Medium — Power of Confusion Matrix](https://dstechbytes.medium.com/a-secret-weapon-power-of-confusion-matrix-in-machine-learning-511c122f24d4)

For a binary classifier:

| | Predicted positive | Predicted negative |
|---|---|---|
| Actual positive | TP | FN |
| Actual negative | FP | TN |

For a multi-class classifier, each row and column represents a class.

- Diagonal cells = Correct predictions.
- Off-diagonal cells = Misclassifications.

**Key point:** A confusion matrix tells you not just how many mistakes occurred, but which classes the model confuses with one another.

---

## 3. CV-Specific Evaluation

Traditional metrics tell us how well the model performs.

Computer vision evaluation goes one step further:

- Which image classes are difficult to recognise?
- Which images are being misclassified?
- Is the model learning the actual object or irrelevant background information?
- Does it fail under particular lighting or camera conditions?
- Does the dataset represent the real-world environment?

These questions help us diagnose and improve a trained model.

### 3.1 Per-Class Performance

Overall accuracy gives us one combined score. Per-class evaluation tells us how well the model performs on each individual category.

Consider a motor defect classification model:

| Class | Precision | Recall | F1-score |
|---|---:|---:|---:|
| Normal | 0.98 | 0.99 | 0.99 |
| Crack | 0.85 | 0.72 | 0.78 |
| Surface defect | 0.91 | 0.88 | 0.89 |
| Deformation | 0.76 | 0.60 | 0.67 |

![Data labeling and annotation reference](https://images.openai.com/static-rsc-4/ckc-FOnAnwWMKsNDffJUzL95GNs53Zn6Mlmb85Xr8v5NReDKliSLFu6ajlUgRGk5rO3O_qmYou1ql7f4MNU_N_zFO7j8viGtBf7Ek3zCVZN4c5J9NCSyGPRw-u9mU4W-vIfQODnhX_diHUxnOYlRqy4NFEJoThQSa67W1Hihug4?purpose=inline)

[Source: Scale — Data Labeling Guide](https://scale.com/guides/data-labeling-annotation-guide)

The example shows why overall accuracy is insufficient.

**Interpretation:**

- Normal has excellent performance.
- Crack has relatively low recall: many actual cracks are missed.
- Surface defect has reasonably balanced precision and recall.
- Deformation has the weakest F1-score.

In a manufacturing inspection system, poor recall for cracks could be particularly dangerous because defective components may pass inspection.

#### How to calculate per-class metrics in Python

Scikit-learn provides `classification_report()`.

```python
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Normal",
            "Crack",
            "Surface Defect",
            "Deformation"
        ]
    )
)
```

**Example output:**

```text
                  precision  recall  f1-score

Normal              0.98     0.99      0.99
Crack               0.85     0.72      0.78
Surface Defect      0.91     0.88      0.89
Deformation         0.76     0.60      0.67

accuracy                              0.92
macro avg           0.88     0.80      0.83
weighted avg        0.92     0.92      0.92
```

*Illustrative values.*

#### Macro vs. Weighted Average

This distinction is important when working with imbalanced datasets.

**Macro average:**

Calculates the metric independently for each class and takes the unweighted average.

$$
F1_{macro}=\frac{1}{C}\sum_{i=1}^{C}F1_i
$$

Every class receives equal importance, regardless of its sample count.

**Weighted average:**

Calculates the average using the number of actual samples in each class as weights.

$$
F1_{weighted}=
\frac{\sum_{i=1}^{C}n_iF1_i}
{\sum_{i=1}^{C}n_i}
$$

Here, \(n_i\) is the number of samples belonging to class \(i\).

**Engineering takeaway:** When your dataset is imbalanced, examine macro F1 and per-class recall alongside accuracy. Weighted F1 alone can hide poor minority-class performance.

---

### 3.2 Misclassified Images

This is one of the most important CV-specific evaluation techniques.

Instead of looking only at metric values, inspect the actual images that the model predicted incorrectly.

For every misclassified image, compare:

1. Original image.
2. Ground-truth label.
3. Predicted label.
4. Prediction confidence.

![Image classification example from Imagenette](https://images.openai.com/static-rsc-4/vHsYCD6h2UD36x3DpSseRuWhQ7_u0Q_SMG_0rTe27tsJsPHkbtrWPiPP6e3dJG3vYoxgS1socOpdyf6R2vOuJM9-o_iUKtW1XUqamEE3AYRR1Jtdpb0Y5Oc5FfhbxwzgSPCxu_FdK09GznpRo_qCWQh73e0KK3uiUmbge8N4Hyg?purpose=inline)

[Source: fastai — Imagenette Tutorial](https://docs.fast.ai/tutorial.imagenette.html)

The image above illustrates a classification model confusing visually different objects, such as a chainsaw and a French horn.

#### Why inspect misclassified images?

Suppose your model predicts:

```text
Actual class: Crack
Predicted class: Normal
Confidence: 94%
```

This is a potentially serious failure.

You should investigate:

- Is the crack too small?
- Is the image blurry?
- Is the lighting poor?
- Is the crack partially hidden?
- Was the image incorrectly labelled?
- Does the training dataset contain enough similar examples?

#### PyTorch: Collect misclassified images

Assume:

- `model` is your trained model.
- `test_loader` returns images and labels.
- `classes` contains the class names.

```python
import torch

model.eval()

misclassified = []

with torch.inference_mode():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs, dim=1
        )

        predictions = probabilities.argmax(dim=1)

        for i in range(len(labels)):

            if predictions[i] != labels[i]:

                misclassified.append({
                    "image": images[i].cpu(),
                    "actual": classes[labels[i].item()],
                    "predicted": classes[predictions[i].item()],
                    "confidence": probabilities[i, predictions[i]].item()
                })

print("Total misclassified:", len(misclassified))
```

#### Display a misclassified image

```python
import matplotlib.pyplot as plt

sample = misclassified[0]

image = sample["image"].permute(1, 2, 0)

plt.imshow(image)

plt.title(
    f"Actual: {sample['actual']}\n"
    f"Predicted: {sample['predicted']}\n"
    f"Confidence: {sample['confidence']:.2%}"
)

plt.axis("off")
plt.show()
```

**Note:** If your images were normalized, reverse the normalization before displaying them to see their original appearance.

**What you should learn:** Misclassified-image inspection converts an abstract metric into a concrete engineering problem.

---

### 3.3 Dataset Bias

Dataset bias occurs when the training dataset does not adequately represent the variety of images the model will encounter in real-world applications.

A model may achieve excellent test accuracy but fail when the image source, environment, or conditions change.

![Dataset bias and distribution shift](https://images.openai.com/static-rsc-4/FvjgWuGN3tA-LL-SClktWwKyzu6TimkW8V3CfZyFghaLUh3yeovsAKPvD7STTHIVnINRCj_fR-CcnjcE7YzAxu-uF7MhKysYmSV7Kl5bWt8-nt9ep-hEHrdQXtstyZUd2qvX7P3hTh_Ecs4rsv1S26bBeezwOa9rLbDBEK1TAG8?purpose=inline)

[Source: MIT — Dataset Bias and Distribution Shift](https://visionbook.mit.edu/bias_and_shift.html)

#### Example: Manufacturing inspection

Imagine you train a defect classification model using images captured under:

- Bright lighting.
- White backgrounds.
- A fixed camera angle.
- One particular manufacturing machine.

Later, the model is deployed in another factory.

The new environment has:

- Dim lighting.
- Different camera angles.
- Different backgrounds.
- Different component finishes.

The model might perform poorly despite having achieved high test accuracy.

#### Types of dataset bias

| Bias type | Example |
|---|---|
| Class imbalance | More normal components than defective components |
| Lighting bias | Training images contain only bright lighting |
| Camera bias | Training images use only one camera |
| Background bias | Defects are always photographed on the same background |
| Source bias | Images come from only one factory |
| Resolution bias | Training images are consistently high-resolution |

#### Background bias — an important example

Suppose every image of a defective component has a red background, while every normal component has a blue background.

The model may learn to associate:

```text
Red background → Defective
Blue background → Normal
```

Instead of learning the actual visual characteristics of the defect.

This is called learning a **spurious correlation**.

The model appears accurate on the test set if the same background pattern exists there, but fails when the background changes.

#### How to identify dataset bias

Evaluate model performance across different subsets of your dataset.

For example:

| Image condition | Accuracy |
|---|---:|
| Bright lighting | 96% |
| Medium lighting | 92% |
| Low lighting | 71% |
| Camera A | 95% |
| Camera B | 78% |
| Camera C | 89% |

These illustrative results reveal that the model struggles under low lighting and with Camera B.

#### How to reduce dataset bias

1. Collect images from different cameras and environments.
2. Include different lighting conditions.
3. Ensure adequate examples of every class.
4. Evaluate each important subgroup separately.
5. Avoid shortcuts such as background or watermark correlations.
6. Test on data collected from a different production environment.

**Important distinction:** A balanced class distribution does not guarantee a representative dataset. You must also consider how and where the images were collected.

---

### 3.4 Failure Case Analysis

A failure case is a situation in which the model produces an incorrect, unreliable, or unexpected prediction.

Misclassified images tell us what went wrong. Failure-case analysis investigates why it went wrong and under which conditions it happens.

#### Common computer vision failure cases

![Artificial weld crack detection](https://images.openai.com/static-rsc-4/PUEMOq3d-TYphvyjtnhOU64cb1goju7-u8l1qWINLaacDVNzAoR0roUHp9dtL5W6PBoYEhFTHQCOU-270fOTLkbgWwTBYkm-s9F8PDcab5EBk-1UCszuBuN8Tgb16YNr61Z2BpV4yxgUHA6gNY_67Atk0HW44lZH411ahGebFdY?purpose=inline)

![Automated optical inspection of reflective parts](https://images.openai.com/static-rsc-4/Lje7Y84kTaJ7a6-trFBArH0Ts9KjK_w5w5kVuY1TJbQfdbul7B6tUcTC73zsia1UxE0MtPWtKQa3uYgGjLZmkMN98Uz65i8Ku-b2FbP-nvyfZwJD16bOVaF45_0YwADYfkelP4ITQpmsAgUb2Gay8k7DWi7NKjKUYG9-gNhqCrM?purpose=inline)

![AI sheet metal defect inspection](https://images.openai.com/static-rsc-4/AaZS0NoFT7bY4D6ZMF98ChTv568Ohvk38tYvsHfTy-vlRRRMvWSKiyhvl1-OR5KCwR_7bBTttK_dZXodn0KSEgxyIULNW4iZ5sKsciS5cOIdMN7MNbKio1Yx_m2R-Yy1gIIyFfdcAgXx8fYf2N7Era1B7OAk0L04srW9wuwxHpQ?purpose=inline)

| Failure case | What happens? | Possible reason |
|---|---|---|
| Poor lighting | Defect is missed | Insufficient visual information |
| Motion blur | Model predicts wrong class | Loss of image details |
| Occlusion | Defect is hidden | Object is partially obstructed |
| Reflection | Reflection is classified as a defect | Visual similarity |
| Unusual orientation | Incorrect classification | Training data lacks orientation variation |
| Low resolution | Small defects are missed | Insufficient image detail |
| Domain shift | Performance drops in a new factory | Different image distribution |

#### Example: Crack detection failure

Suppose the model correctly identifies large cracks but misses tiny cracks.

![Surface defects versus structural defects in steel coils](https://images.openai.com/static-rsc-4/l2x0FBOwOWIl8HXB88EmVxUdJkY0NHDOt_oCq3-Ti6g2qvHSyRcV-CXDikDhYtHS2QPmFZC7syL1RBxjptcQ8xb27Fa0ZTEChzIw0HgAtpVqzHLt_FHXhLwqjqm9hifPbwylOa26jhFnjnGi7T6QN0jN_FRznteVDCGpFv7TT0U?purpose=inline)

[Source: Collier Metals — Surface Defects vs. Structural Defects](https://www.colliermetals.com/blog/2026/2/13/surface-defects-vs-structural-defects-in-steel-coils)

**Possible investigation:**

1. Check the resolution of the input images.
2. Examine whether small cracks are adequately represented in training.
3. Check whether resizing to 224 × 224 removes important details.
4. Inspect whether augmentation introduces excessive blur or cropping.
5. Evaluate recall specifically for small cracks.

**Possible improvements:**

- Collect more small-crack examples.
- Use higher-resolution input images.
- Adjust preprocessing and cropping.
- Train with suitable scale augmentation.
- Consider a detection or segmentation approach if localisation is required.

#### Failure analysis workflow

![Improving vision model performance workflow](https://images.openai.com/static-rsc-4/98HhSGB6q0ODsqV-iaR7ahXTfGVr2zBPgfUqi0RdX7SiEeTFEjeXPR3b6QWA69jLW8f7Vp38w76a5LszdtJRh4QgalGm1P2gOBJuMTcQWaguS5E3uSe5ob1apPKWlVUAeR9VLT3XIc_MP2rw1dC-oRHWAT6FGCyj4qD5DOP3FaQ?purpose=inline)

[Source: Edge AI and Vision Alliance — Improving Vision Model Performance](https://www.edge-ai-vision.com/2025/01/improving-vision-model-performance-using-roboflow-and-tenyks/)

A practical workflow:

1. Collect incorrect predictions.
2. Group them by failure condition.
3. Identify recurring patterns.
4. Determine whether the issue is related to data, labels, preprocessing, or model capacity.
5. Make a targeted improvement.
6. Retrain and evaluate using the same held-out test set.

**Do not automatically retrain whenever you see a mistake. First identify its underlying cause.**

---

## 4. Complete Evaluation Pipeline in PyTorch

Let's combine the concepts into one evaluation workflow.

### Step 1 — Generate predictions

```python
import torch

model.eval()

all_predictions = []
all_labels = []

with torch.inference_mode():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().tolist()
        )

        all_labels.extend(
            labels.tolist()
        )
```

### Step 2 — Calculate evaluation metrics

```python
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("Accuracy:", accuracy)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=classes,
        zero_division=0
    )
)
```

### Step 3 — Visualise the confusion matrix

```python
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Image Classification Confusion Matrix")

plt.show()
```

![Confusion matrix and classification visual reference](https://images.openai.com/static-rsc-4/jJbcD8eEMLh_hdM0SmioA8MHYBIN4nTKekZwzZ31bzHU6v4n0wB1upHYYHbH3nS_nCsRFr3kQPUcGc1VX3UDM2zlAZr6rB9i9tk_VEFtZO0-PmB0PfGXAbGbB4qEq8z_qyjzW3R-jb_ypKgKB0NQwyb5ZdpzydibDn7AMdUqouM?purpose=inline)

[Source: Znews.vn](https://znews.vn/ai-giai-captcha-nhanh-hon-con-nguoi-post1500710.html)

**Note:** Scikit-learn's `confusion_matrix` uses actual classes as rows and predicted classes as columns. The diagonal contains correct predictions. Normalising by rows gives per-class recall proportions. Other tools may use the opposite axis convention, so always check the labels.

### Step 4 — Inspect the failure cases

Use the misclassified-image collection from Section 3.2.

Look for:

- Repeated class confusion.
- Small or hidden defects.
- Unusual lighting.
- Incorrect annotations.
- Similar-looking categories.

### Step 5 — Perform subgroup evaluation

Calculate metrics separately for:

- Lighting conditions.
- Camera sources.
- Production batches.
- Image resolutions.
- Defect sizes, if available.

This helps reveal weaknesses hidden by the overall test score.

---

## 5. Overall Evaluation vs. CV-Specific Evaluation

| Evaluation approach | What it tells you |
|---|---|
| Accuracy | Overall correctness |
| Precision | Reliability of positive predictions |
| Recall | Ability to identify actual positive samples |
| F1-score | Balance between precision and recall |
| Confusion matrix | Which classes are confused |
| Per-class performance | Which categories perform poorly |
| Misclassified images | What incorrect predictions look like |
| Dataset bias | Whether data limitations affect performance |
| Failure analysis | Under what conditions the model fails |

**For further reading:**

- [Computer Vision — Evaluating a Classifier](https://sbirchfield.github.io/cvintro/lessons/lesson36_evaluating_a_classifier.html)
- [AWS — Understanding Confusion Matrices](https://docs.aws.amazon.com/rekognition/latest/customlabels-dg/im-confusion-matrix.html)
- [MIT — Dataset Bias and Distribution Shift](https://visionbook.mit.edu/bias_and_shift.html)
- [Fraunhofer — Robustness Evaluation of AI Models](https://robuscope.iks.fraunhofer.de/evaluate-predictions)

---

## 6. What You Should Remember 🧠

1. Accuracy alone does not describe the complete performance of a classifier.
2. Per-class precision, recall, and F1 reveal weaknesses in individual categories.
3. Misclassified-image inspection helps identify actual visual problems.
4. Dataset bias can cause a model to learn irrelevant patterns.
5. Failure-case analysis identifies weaknesses such as blur, poor lighting, occlusion, and domain shift.
6. A good CV evaluation process combines numerical metrics with visual inspection.
7. Always evaluate on genuinely unseen data representative of the deployment environment.

### The engineering mindset

> Don't just ask how accurate your model is. Ask which images it gets wrong, why it gets them wrong, and whether those mistakes matter in the real world.

